In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import requests
import json

## PASO 1: 

Rellenar programáticamente el selector de tipo "Registro" y un rango de fechas, ejecutar la búsqueda y capturar la URL resultante para procesarla o iterar sobre múltiples rangos de fechas de forma automática.

> El parámetro s= es un session/search ID — el frontend Vue guarda la búsqueda en el servidor y te regresa un UUID. 

> 1.1 Cada busqueda Genera un ID "SearchId" (De busqueda), en este paso tenemos que replicar la llamada API que crea ese ID. Para esto hay que identificar el flujo de request que utiliza la API (pagina >se comunica> con servidor).

    En este caso, el flujo es:
    -> Record, registra la busqueda (query parametres)
    -> Result, pagina con los registros

### Base de la funcion llamar API

In [ ]:
# session = requests.Session()
# session.get("https://marcia.impi.gob.mx/marcas/search/quick")

# # Extraes el token de las cookies que ya tienes
# xsrf_token = session.cookies.get("XSRF-TOKEN")
# print("Token:", xsrf_token)

# url = "https://marcia.impi.gob.mx/marcas/search/internal/record"

# headers = {
#     "X-XSRF-TOKEN": xsrf_token,       
#     "Content-Type": "application/json",
#     "Referer": "https://marcia.impi.gob.mx/marcas/search/quick",
# }

# payload = {
#     "_type": "Search$Structured",
#     "images": [],
#     "query": {
#         "_type": "Search$Structured",
#         "number": None,
#         "title": None,
#         "titleOption": None,
#         "name": None,
#         "status": None,
#         "appType": None,
#         "classes": None,
#         "codes": None,
#         "goodsAndServices": None,
#         "indicators": None,
#         "markType": None,
#         "wordSet": None,
#         "date": {
#             "types": ["DATE_REGISTRATION"],
#             "date": {
#                 "from": "2023-01-01",
#                 "to": "2023-01-19"
#             }
#         }
#     }
# }

# respuesta = session.post(url, json=payload, headers=headers)
# print(respuesta.status_code)
# print(respuesta.text[:500])  

## Set de funciones para (llamar api en bulk)

MARCIA tiene un límite de 10,000 resultados por búsqueda. Un año completo tiene muchos más. Entonces necesitas partir el año en pedazos donde cada pedazo tenga menos de 10,000 registros, y guardar la URL de cada pedazo.

In [4]:
# ── Sesión y autenticación ──────────────────────────────────────
session = requests.Session()
session.get("https://marcia.impi.gob.mx/marcas/search/quick")

xsrf_token = session.cookies.get("XSRF-TOKEN")
print("Token:", xsrf_token)

url = "https://marcia.impi.gob.mx/marcas/search/internal/record"

headers = {
    "X-XSRF-TOKEN": xsrf_token,
    "Content-Type": "application/json",
    "Referer": "https://marcia.impi.gob.mx/marcas/search/quick",
}

# ── Función 1: llamada a la API, funcion inicial
def llamar_api(from_date, to_date):
    payload = {
        "_type": "Search$Structured",
        "images": [],
        "query": {
            "_type": "Search$Structured",
            "number": None,
            "title": None,
            "titleOption": None,
            "name": None,
            "status": None,
            "appType": None,
            "classes": None,
            "codes": None,
            "goodsAndServices": None,
            "indicators": None,
            "markType": None,
            "wordSet": None,
            "date": {
                "types": ["DATE_REGISTRATION"],
                "date": {
                    "from": from_date.strftime("%Y-%m-%d"),
                    "to":   to_date.strftime("%Y-%m-%d")
                }
            }
        }
    }
    respuesta = session.post(url, json=payload, headers=headers)
    return respuesta.json()

#  Función 2: encontrar todos los rangos válidos de un año 
def encontrar_rangos(año):
    resultados = {}
    contador   = 1
    ultima_valida = None

    from_date = date(año, 1, 1)
    to_date   = date(año, 1, 1)
    fin_año   = date(año, 12, 31)

    while to_date <= fin_año:
        datos = llamar_api(from_date, to_date)
        count = datos["count"]
        print(f"  {from_date} → {to_date} | count: {count}")

        if count > 10000:
            resultados[f"llamada_{contador}"] = ultima_valida
            contador  += 1
            from_date  = to_date                  
            ultima_valida = None        

        else:
            search_id = datos["id"]
            ultima_valida = {
                "from":      from_date.strftime("%Y-%m-%d"),
                "to":        to_date.strftime("%Y-%m-%d"),
                "count":     count,
                "search_id": search_id,
                "url":       f"https://marcia.impi.gob.mx/marcas/search/result?s={search_id}&m=l"
            }
            to_date = to_date + timedelta(days=1)  # avanzar el cursor

    if ultima_valida:
        resultados[f"llamada_{contador}"] = ultima_valida

    return resultados

# ── Ejecución 
rangos = encontrar_rangos(2023)

print("\n── Rangos encontrados ──────────────────────────────")
for key, val in rangos.items():
    print(f"{key}: {val['from']} → {val['to']} | count: {val['count']}")
    print(f"       {val['url']}\n")


with open("1.rangos_busquedas.json", "w", encoding="utf-8") as f:
    json.dump(rangos, f, ensure_ascii=False, indent=2)

Token: 090a9778-fa31-474b-ba3f-357db328e759
  2023-01-01 → 2023-01-01 | count: 0
  2023-01-01 → 2023-01-02 | count: 646
  2023-01-01 → 2023-01-03 | count: 1480
  2023-01-01 → 2023-01-04 | count: 2222
  2023-01-01 → 2023-01-05 | count: 3014
  2023-01-01 → 2023-01-06 | count: 3549
  2023-01-01 → 2023-01-07 | count: 3549
  2023-01-01 → 2023-01-08 | count: 3549
  2023-01-01 → 2023-01-09 | count: 4401
  2023-01-01 → 2023-01-10 | count: 4814
  2023-01-01 → 2023-01-11 | count: 5023
  2023-01-01 → 2023-01-12 | count: 5885
  2023-01-01 → 2023-01-13 | count: 6378
  2023-01-01 → 2023-01-14 | count: 6378
  2023-01-01 → 2023-01-15 | count: 6378
  2023-01-01 → 2023-01-16 | count: 7299
  2023-01-01 → 2023-01-17 | count: 8198
  2023-01-01 → 2023-01-18 | count: 8795
  2023-01-01 → 2023-01-19 | count: 9263
  2023-01-01 → 2023-01-20 | count: 10016
  2023-01-20 → 2023-01-20 | count: 753
  2023-01-20 → 2023-01-21 | count: 753
  2023-01-20 → 2023-01-22 | count: 753
  2023-01-20 → 2023-01-23 | count: 1555
  